In [ ]:
import os
import requests
import pandas as pd

URL = "https://77.rosstat.gov.ru/storage/mediabank/Динамика%20индекса%20потребительских%20цен%20на%20товары%20и%20услуги%20в%202000-2026%20гг..xlsx"
RAW_PATH = "data/raw/cpi.xlsx"
PROCESSED_PATH = "data/processed/cpi_cleaned.csv"

def download_file():
    os.makedirs("data/raw", exist_ok=True)
    r = requests.get(URL, verify=False)  # отключаем проверку SSL
    r.raise_for_status()
    with open(RAW_PATH, "wb") as f:
        f.write(r.content)
    print("Файл скачан")

def preprocess():
    os.makedirs("data/processed", exist_ok=True)
    df = pd.read_excel(RAW_PATH, skiprows=3)
    df = df.rename(columns={df.columns[0]: "month"})
    df = df.dropna(subset=["month"])
    df = df.melt(id_vars=["month"], var_name="year", value_name="value")
    df = df.dropna(subset=["value"])
    df["value"] = df["value"].astype

In [ ]:
!pip install pandas openpyxl requests

In [ ]:
import requests

URL = "https://77.rosstat.gov.ru/storage/mediabank/Динамика%20индекса%20потребительских%20цен%20на%20товары%20и%20услуги%20в%202000-2026%20гг..xlsx"

r = requests.get(URL, verify=False)
r.raise_for_status()

with open("cpi.xlsx", "wb") as f:
    f.write(r.content)

print("Файл скачан")

In [ ]:
import pandas as pd

df = pd.read_excel("cpi.xlsx", skiprows=3)
df.head()

In [ ]:
df = df.rename(columns={"Unnamed: 0": "month"})

In [ ]:
df = df.dropna(subset=["month"])

In [ ]:
df_long = df.melt(
    id_vars=["month"],
    var_name="year",
    value_name="cpi"
)

In [ ]:
df_long = df_long.dropna(subset=["cpi"])

In [ ]:
df_long[df_long["cpi"].astype(str).str.contains(r"[^\d,\.]")].head()

In [ ]:
df_long["cpi"] = (
    df_long["cpi"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace(r"[^\d\.]", "", regex=True)
)

In [ ]:
df_long["cpi"] = pd.to_numeric(df_long["cpi"], errors="coerce")

In [ ]:
df_long = df_long.dropna(subset=["cpi"])

In [ ]:
df_long.info()
df_long.head()

In [ ]:
print(df_long[['month', 'month_num']].head(20))
print(df_long['month_num'].unique())
print(df_long['year'].unique())

In [ ]:
# словарь для русских месяцев
month_map = {
    'январь': 1,
    'февраль': 2,
    'март': 3,
    'апрель': 4,
    'май': 5,
    'июнь': 6,
    'июль': 7,
    'август': 8,
    'сентябрь': 9,
    'октябрь': 10,
    'ноябрь': 11,
    'декабрь': 12
}

# переводим все в нижний регистр
df_long['month_lower'] = df_long['month'].str.lower()

# создаём числовой столбец
df_long['month_num'] = df_long['month_lower'].map(month_map)

# проверка
print(df_long[['month', 'month_num']].head(15))

In [ ]:
print(df_long.columns)

In [ ]:
df_long.to_csv("cpi_cleaned.csv", index=False)
df = pd.read_csv("cpi_cleaned.csv")

In [ ]:
# 4. Средний CPI по годам
df_year = df_long.groupby('year')['cpi'].mean().reset_index()

plt.figure(figsize=(10,5))
plt.plot(df_year['year'], df_year['cpi'], marker='o')
plt.title('Средний индекс потребительских цен по годам')
plt.xlabel('Год')
plt.ylabel('CPI')
plt.grid(True)
plt.show()

In [ ]:
# 5. Сезонная динамика CPI
df_plot = df_long.dropna(subset=['month_num'])
plt.figure(figsize=(12,6))
sns.lineplot(data=df_plot, x='month_num', y='cpi', hue='year', palette='tab20')
plt.title('Сезонная динамика CPI по месяцам')
plt.xlabel('Месяц')
plt.ylabel('CPI')
plt.show()

In [ ]:
# 6. Годовой прирост CPI (инфляция)
df_year['cpi_growth'] = df_year['cpi'].pct_change() * 100

plt.figure(figsize=(10,5))
plt.bar(df_year['year'], df_year['cpi_growth'])
plt.title('Годовой прирост CPI, %')
plt.xlabel('Год')
plt.ylabel('Рост, %')
plt.grid(True)
plt.show()

1. Средний CPI по годам показывает общий рост цен, с резкими скачками в кризисные периоды:
   - 2008–2009: мировой финансовый кризис
   - 2014–2015: падение рубля и рост цен на импортные товары
   - 2020: пандемия COVID-19

2. Сезонная динамика показывает небольшие колебания:
   - Летом и зимой цены выше, возможно из-за сезонных продуктов и коммунальных услуг
   - Весной и осенью рост цен более плавный

3. Годовой прирост CPI (инфляция) неравномерен:
   - Пики совпадают с кризисными годами
   - В стабильные периоды рост умеренный (2–5%)

4. Гипотезы:
   - Резкие скачки CPI связаны с девальвацией рубля, внешними экономическими шоками или ростом цен на импортные товары
   - Плавный рост отражает нормальное повышение цен на базовые товары и услуги
"""